# 4회차 — Qwen JSON 응답과 로봇 센서 데이터 검증
**대상:** 로봇공학과 2학년 · **환경:** CPU / VS Code Jupyter / 105분

**목표:** Qwen으로 JSON 응답을 생성하고, Python에서 파싱·검증한 뒤 가상 센서 데이터에 적용합니다. **실제 로봇을 구동하지 않습니다.**

> 첫 실행에는 Hugging Face 모델 다운로드를 위한 인터넷 연결이 필요합니다. 모델은 `Qwen/Qwen2.5-0.5B-Instruct`를 사용합니다. 모든 셀을 위에서 아래로 실행하세요.

## 1. 실행 환경 확인 (10분)

In [ ]:
# Python 실행 환경 확인
import sys

# JSON, CSV, 실행 시간 측정 라이브러리 불러오기
import json
import csv
import time
from pathlib import Path

# PyTorch와 Transformers 불러오기
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 현재 커널의 Python 경로와 버전 출력
print("Python:", sys.version.split()[0])
print("경로:", sys.executable)
print("PyTorch:", torch.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())

## 2. Qwen 모델 CPU 로딩 (15분)
처음 실행 시 모델 파일을 다운로드합니다. 이후에는 로컬 캐시를 재사용합니다.

In [ ]:
# CPU에서 실행할 Qwen 모델 이름 지정
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# 모델 로딩 시작 시간 기록
start = time.perf_counter()

# 토크나이저 불러오기
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 모델을 CPU에 불러오기 (Transformers 버전에 따라 dtype 사용)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float32).to("cpu")

# 학습이 아닌 추론 모드로 변경
model.eval()

# 모델 정보 출력
print("모델 로딩 완료")
print("장치:", next(model.parameters()).device)
print(f"로딩 시간: {time.perf_counter() - start:.2f}초")

## 3. JSON과 Python 딕셔너리 이해 (15분)
JSON은 문자열 형식의 데이터이고, `json.loads()`는 이를 Python 객체로 변환합니다.

In [ ]:
# JSON 형식의 문자열 준비
json_text = '{"direction": "left", "speed": 0.2, "reason": "왼쪽 공간이 넓음"}'

# JSON 문자열을 Python 딕셔너리로 변환
robot_command = json.loads(json_text)

# 데이터 형식과 항목 확인
print(type(robot_command))
print("이동 방향:", robot_command["direction"])
print("속도:", robot_command["speed"])

# 딕셔너리를 보기 좋은 JSON 문자열로 변환
print(json.dumps(robot_command, ensure_ascii=False, indent=2))

## 4. Qwen에게 JSON 출력 요청하기 (15분)
프롬프트에서 허용된 필드와 자료형을 명확히 지정합니다. 모델이 항상 올바른 JSON을 생성하는 것은 아닙니다.

In [ ]:
# Qwen에게 전달할 가상 센서 값
sensor = {"front_cm": 20, "left_cm": 80, "right_cm": 35}

# 모델에게 JSON 출력 형식과 제한 조건 설명
system_prompt = ("당신은 로봇 센서 데이터를 읽는 교육용 시뮬레이터입니다. "
                 "응답은 반드시 JSON 객체 하나만 출력하세요. "
                 "키는 direction, speed, reason 세 개만 사용하세요. "
                 "direction은 stop, left, right, forward 중 하나입니다. "
                 "speed는 0부터 0.3 사이 숫자입니다. "
                 "코드 블록 표시나 추가 설명은 넣지 마세요.")

# 센서 데이터를 사용자 메시지로 구성
user_prompt = f"가상 센서 데이터: {json.dumps(sensor, ensure_ascii=False)}. 안전한 이동 명령을 제안하세요."

# Qwen 대화 메시지 작성
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

# Qwen 전용 대화 템플릿 적용
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
)

# 기울기 계산 없이 모델 응답 생성
with torch.inference_mode():
    outputs = model.generate(
        inputs, max_new_tokens=160, do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

# 입력 토큰을 제외하고 새로 생성한 토큰만 디코딩
raw_response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

# 모델이 실제로 반환한 원문 확인
print("Qwen 원문 응답:")
print(raw_response)

## 5. JSON 파싱과 오류 처리 (15분)
응답이 JSON 형식이 아니면 `JSONDecodeError`가 발생할 수 있습니다. 오류를 숨기지 않고 기록합니다.

In [ ]:
# 모델 응답의 JSON 파싱 시도
try:
    parsed_response = json.loads(raw_response)
    print("JSON 파싱 성공:", parsed_response)
except json.JSONDecodeError as error:
    parsed_response = None
    print("JSON 파싱 실패:", error)
    print("원문을 확인하고 프롬프트를 수정하세요.")

## 6. 출력 데이터 검증 (15분)
**주의:** JSON 파싱 성공은 데이터가 안전하거나 정확하다는 뜻이 아닙니다. 허용 키·자료형·범위를 별도로 검사합니다.

In [ ]:
# 허용되는 이동 방향 정의
ALLOWED_DIRECTIONS = {"stop", "left", "right", "forward"}

# 명령의 형식과 값을 검증하는 함수 정의
def validate_command(command):
    # 딕셔너리인지 확인
    if not isinstance(command, dict):
        return False, "JSON 객체가 아닙니다."

    # 필수 키와 불필요한 키 확인
    if set(command.keys()) != {"direction", "speed", "reason"}:
        return False, "필드 이름이 올바르지 않습니다."

    # 이동 방향이 허용 목록에 있는지 확인
    if command["direction"] not in ALLOWED_DIRECTIONS:
        return False, "허용되지 않은 이동 방향입니다."

    # 속도가 숫자인지 확인 (bool은 숫자로 인정하지 않음)
    if type(command["speed"]) not in (int, float):
        return False, "속도는 숫자여야 합니다."

    # 속도 범위 검사 (NaN도 거부)
    if not 0 <= command["speed"] <= 0.3:
        return False, "속도가 허용 범위를 벗어났습니다."

    # 이유가 비어 있지 않은 문자열인지 확인
    if not isinstance(command["reason"], str) or not command["reason"].strip():
        return False, "판단 이유가 올바르지 않습니다."

    # 정지 명령이면 속도가 반드시 0이어야 함
    if command["direction"] == "stop" and command["speed"] != 0:
        return False, "정지 명령의 속도는 0이어야 합니다."

    # 모든 검사를 통과한 경우
    return True, "검증 통과"

# 파싱에 성공한 경우에만 검증 실행
if parsed_response is not None:
    valid, message = validate_command(parsed_response)
    print("검증 결과:", valid, message)
else:
    print("파싱 실패로 검증을 건너뜁니다.")

## 7. 가상 센서 데이터에 안전 규칙 적용 (15분)
LLM의 출력은 **제안**일 뿐입니다. 아래 코드는 독립적인 규칙으로 시뮬레이션 명령을 결정합니다. 실제 로봇의 안전 제어에는 사용할 수 없습니다.

In [ ]:
# 가상 센서 데이터 세 건 준비
sensor_samples = [
    {"front_cm": 20, "left_cm": 80, "right_cm": 35},
    {"front_cm": 70, "left_cm": 40, "right_cm": 50},
    {"front_cm": 12, "left_cm": 10, "right_cm": 15},
]

# 센서 데이터의 유효성 확인 함수 정의
def validate_sensor(data):
    # 필수 센서 항목 확인
    if set(data.keys()) != {"front_cm", "left_cm", "right_cm"}:
        return False

    # 센서 거리의 자료형과 허용 범위 확인
    return all(type(value) in (int, float) and 0 <= value <= 500 for value in data.values())

# 독립적인 규칙 기반 명령 생성 함수 정의
def rule_based_command(data):
    # 센서 데이터가 잘못되면 정지
    if not validate_sensor(data):
        return {"direction": "stop", "speed": 0.0, "reason": "센서 값 오류"}

    # 전방 거리가 충분하면 전진
    if data["front_cm"] >= 50:
        return {"direction": "forward", "speed": 0.2, "reason": "전방 공간 충분"}

    # 좌우 모두 가까우면 정지
    if max(data["left_cm"], data["right_cm"]) < 30:
        return {"direction": "stop", "speed": 0.0, "reason": "안전한 회피 공간 부족"}

    # 좌측 공간이 더 넓으면 좌회전
    if data["left_cm"] >= data["right_cm"]:
        return {"direction": "left", "speed": 0.1, "reason": "좌측 공간이 더 넓음"}

    # 그 외에는 우회전
    return {"direction": "right", "speed": 0.1, "reason": "우측 공간이 더 넓음"}

# 모든 가상 센서 데이터에 규칙 적용
for data in sensor_samples:
    command = rule_based_command(data)
    print("센서:", data, "→ 명령:", command)

## 8. CSV 저장 및 결과 확인 (5분)

In [ ]:
# CSV 파일에 저장할 결과 리스트 생성
rows = []

# 센서 데이터와 검증된 규칙 기반 결과를 한 행씩 구성
for data in sensor_samples:
    command = rule_based_command(data)
    rows.append({**data, **command})

# Notebook 실행 폴더에 CSV 파일 저장
output_path = Path("04_Robot_Results.csv")

# CSV를 UTF-8 BOM 인코딩으로 저장해 Excel 한글 깨짐 방지
with output_path.open("w", newline="", encoding="utf-8-sig") as file:
    writer = csv.DictWriter(file, fieldnames=[
        "front_cm", "left_cm", "right_cm", "direction", "speed", "reason"
    ])
    writer.writeheader()
    writer.writerows(rows)

# 저장된 파일 경로와 행 수 출력
print("저장 완료:", output_path.resolve())
print("저장된 데이터:", len(rows), "건")

## 9. 학생 확인 문제 및 정리

1. `json.loads()`와 `json.dumps()`의 차이는 무엇인가?
2. 모델이 JSON 문자열을 출력했더라도 검증이 필요한 이유는 무엇인가?
3. `try-except`를 사용하는 이유는 무엇인가?
4. 로봇 제어에서 LLM 출력과 독립적인 안전 규칙을 분리해야 하는 이유는 무엇인가?

**제출:** `04_Qwen_JSON_Practice.ipynb`, `04_Robot_Results.csv`

**GitHub 업로드 (기존 저장소가 연결된 PC에서만):**
```powershell
git add 04_Qwen_JSON_Practice.ipynb 04_Robot_Results.csv
git commit -m "4회차 Qwen JSON 및 로봇 센서 실습"
git push
```